# 02. Metrics Development

This notebook develops and validates the five core inventory health metrics that power the Retail Decision AI scoring engine: `recent_sales_7d` (short-horizon demand), `days_of_stock_remaining` (depletion urgency), `reorder_gap` (distance from reorder threshold), `demand_pressure_score` (rate of stock draw-down), and `stockout_risk_score` (composite risk signal). Each metric is built incrementally on a merged inventory-sales base, with logic verified against expected edge cases before being handed off. The final, validated formulas are productionised in `engine/metrics.py` and `engine/scoring.py`, which are the authoritative source for all downstream pipeline layers including the AI dashboard.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from utils.data_loader import load_all_data

sales_df, products_df, inventory_df = load_all_data()

print("Sales shape:", sales_df.shape)
print("Products shape:", products_df.shape)
print("Inventory shape:", inventory_df.shape)

**`recent_sales_7d`**

Captures the total units sold per store-product pair in the 7 days leading up to the latest date in the dataset -- a short-horizon demand signal used to detect whether current stock is being drawn down quickly.

In [ ]:
sales = sales_df.copy()

latest_date = sales["date"].max()

recent_7d_start = latest_date - pd.Timedelta(days=7)

recent_sales = (
    sales[sales["date"] > recent_7d_start]
    .groupby(["store_nbr", "item_nbr"])["unit_sales"]
    .sum()
    .reset_index()
)

recent_sales = recent_sales.rename(columns={
    "store_nbr": "store_id",
    "item_nbr": "product_id",
    "unit_sales": "recent_sales_7d"
})

print("Shape:", recent_sales.shape)
display(recent_sales.head())

**Merge to Inventory Base**

A left merge preserves all store-product combinations in the inventory base, filling `recent_sales_7d` with zero for pairs that had no recorded sales in the window.

In [ ]:
inventory_base = inventory_df.copy()

merged = inventory_base.merge(
    recent_sales,
    on=["store_id", "product_id"],
    how="left"
)

merged["recent_sales_7d"] = merged["recent_sales_7d"].fillna(0)

print("Shape:", merged.shape)
display(merged.head())

In [ ]:
merged.recent_sales_7d.isnull().sum()

**`days_of_stock_remaining`**

Estimates how many days the current stock level will last at the observed average daily sales rate -- the primary urgency signal for restock decisions.

In [ ]:
avg_sales = (
    sales_df
    .groupby(["store_nbr", "item_nbr"])["unit_sales"]
    .mean()
    .reset_index()
    .rename(columns={
        "store_nbr": "store_id",
        "item_nbr": "product_id",
        "unit_sales": "avg_daily_sales"
    })
)

full_pairs = merged[["store_id", "product_id"]].copy()

avg_sales_full = full_pairs.merge(
    avg_sales,
    on=["store_id", "product_id"],
    how="left"
)

avg_sales_full["avg_daily_sales"] = avg_sales_full["avg_daily_sales"].fillna(0)

print("Shape:", avg_sales_full.shape)
display(avg_sales_full.head())
print("NaN avg_daily_sales:", avg_sales_full["avg_daily_sales"].isnull().sum())

**Merge `avg_daily_sales` to `merged`**

`avg_daily_sales` is derived from the full historical sales window (2016-2017) grouped by store and product, giving a stable baseline demand estimate that smooths out short-term spikes.

In [ ]:
merged = merged.merge(
    avg_sales_full,
    on=["store_id", "product_id"],
    how="left"
)

print("Shape:", merged.shape)
display(merged.head())
print("NaN avg_daily_sales:", merged["avg_daily_sales"].isnull().sum())

**Calculate `days_of_stock_remaining`**

Zero-demand items are assigned a sentinel value of 999 to indicate no meaningful depletion risk, preventing division-by-zero and keeping them at the bottom of any urgency ranking.

In [ ]:
import numpy as np

merged["days_of_stock_remaining"] = np.where(
    merged["avg_daily_sales"] > 0,
    merged["stock_level"] / merged["avg_daily_sales"],
    999  # capped value
)

print("Shape:", merged.shape)
display(
    merged[
        [
            "product_id",
            "store_id",
            "stock_level",
            "avg_daily_sales",
            "days_of_stock_remaining"
        ]
    ].head()
)

print("Numbers of infinite values:", np.isinf(merged["days_of_stock_remaining"]).sum())

**`reorder_gap`**

A positive `reorder_gap` means stock has already fallen below the reorder point and replenishment is overdue; a negative value means there is still buffer before the reorder threshold is breached.

In [ ]:
merged["reorder_gap"] = merged["reorder_point"] - merged["stock_level"]

print("Shape:", merged.shape)
display(
    merged[
        [
            "product_id",
            "store_id",
            "stock_level",
            "reorder_point",
            "reorder_gap"
        ]
    ].head()
)

print("Min reorder_gap:", merged["reorder_gap"].min())
print("Max reorder_gap:", merged["reorder_gap"].max())
print("Jumlah row reorder_gap > 0:", (merged["reorder_gap"] > 0).sum())

**`demand_pressure_score`**

Measures how fast recent demand is drawing down current stock -- a high pressure score means the stock is being consumed quickly relative to what is on hand, even if the absolute stock level still looks safe.

In [ ]:
merged["demand_pressure"] = np.where(
    (merged["stock_level"] > 0) & (merged["recent_sales_7d"] > 0),
    merged["recent_sales_7d"] / merged["stock_level"],
    0
)

merged["is_zero_demand"] = (merged["avg_daily_sales"] == 0).astype(int)
 
print("Shape:", merged.shape)

display(
    merged[
        [
            "product_id",
            "store_id",
            "stock_level",
            "recent_sales_7d",
            "demand_pressure"
        ]
    ].head()
)

print("Max demand_pressure:", merged["demand_pressure"].max())
print("Jumlah high pressure (>1):", (merged["demand_pressure"] > 1).sum())

**`stockout_risk_score`**

A composite score combining three signals: whether stock has breached its reorder point (weighted x2), short-term demand pressure, and historical stockout frequency -- zero-demand items are excluded from risk scoring entirely.

In [ ]:
merged["stockout_risk_score"] = np.where(
    merged["is_zero_demand"] == 1,
    0,  # NO RISK
    (merged["reorder_gap"] > 0).astype(int) * 2
    + merged["demand_pressure"]
    + merged["stockout_frequency"]
)

print("Shape:", merged.shape)

display(
    merged[
        [
            "product_id",
            "store_id",
            "reorder_gap",
            "demand_pressure",
            "stockout_frequency",
            "stockout_risk_score"
        ]
    ].head()
)

print("Max risk:", merged["stockout_risk_score"].max())
print("Top 5 highest risk:")
display(merged.sort_values("stockout_risk_score", ascending=False).head())

# STEP 6 -- Priority Engine

The metrics computed above are fed into a priority classification layer that translates continuous risk scores into four actionable levels -- CRITICAL, HIGH, MEDIUM, LOW -- using data-driven quantile thresholds rather than fixed cutoffs.

## Priority Engine -- Exploratory Decision Layer

At this stage, the inventory health metrics and risk signals are translated into operational priority labels. The goal is to identify which store-product combinations need the closest attention so the business does not miss restock windows.

The priority engine draws on the following signals:
- `recent_sales_7d` -- short-horizon demand activity
- `avg_daily_sales` -- stable baseline demand
- `stock_level` -- current inventory position
- `reorder_point` and `reorder_gap` -- stock position relative to the safe restock threshold
- `demand_pressure` -- rate at which stock is being drawn down
- `stockout_frequency` -- historical pattern of stockout events

From these inputs the engine produces:
- `priority_score` -- continuous composite risk value
- `priority_rank` -- dense rank across all store-product pairs
- `priority_level` -- categorical label: CRITICAL / HIGH / MEDIUM / LOW
- `needs_action` -- boolean flag for items requiring operational attention

A key insight from this exploration is that high-risk items are not always items that have already crossed the static reorder threshold. Some items still appear safe by threshold alone but show elevated demand pressure -- which is why the scoring model incorporates demand behaviour rather than relying solely on reorder point logic.

**Note:** this notebook serves as the exploratory and sanity-check layer. The validated logic is productionised in `engine/metrics.py` and `engine/scoring.py` for stable, repeatable use by the dashboard.

In [ ]:
# =========================
# STEP 6 — PRIORITY ENGINE
# =========================

# 1) Base score
merged["priority_score"] = merged["stockout_risk_score"]

# 2) Dynamic thresholds from data
q90 = merged["priority_score"].quantile(0.9)
q70 = merged["priority_score"].quantile(0.7)

# 3) Priority labeling
def assign_priority(score):
    if score >= q90:
        return "CRITICAL"
    elif score >= q70:
        return "HIGH"
    elif score >= 1:
        return "MEDIUM"
    else:
        return "LOW"

merged["priority_level"] = merged["priority_score"].apply(assign_priority)

# 4) Ranking
merged["priority_rank"] = merged["priority_score"].rank(
    ascending=False,
    method="dense"
)

# 5) Decision flag
merged["needs_action"] = merged["priority_level"].isin(
    ["CRITICAL", "HIGH", "MEDIUM"]
)

# 6) Validation
display(
    merged.sort_values(["priority_score", "priority_rank"], ascending=[False, True]).head(10)
)

In [ ]:
print(merged["priority_level"].value_counts())
print(merged["needs_action"].value_counts())
print("q90:", q90)
print("q70:", q70)

# Validation & Export

This section validates that the exploratory logic above has been correctly productionised in `engine/metrics.py`, `engine/scoring.py`, and `build_executive_view`, then consolidates and exports the final enriched output.

## **`metrics.py`**

Confirms that `load_and_build_metrics_df()` reproduces the same metric columns and score distributions as the exploratory derivation above.

In [ ]:
from engine.metrics import load_and_build_metrics_df

metrics_df = load_and_build_metrics_df()

print(metrics_df.shape)
display(metrics_df.head())

In [ ]:
print(metrics_df.columns)

In [ ]:
metrics_df["stockout_risk_score"].describe()

In [ ]:
display(
    metrics_df.sort_values("stockout_risk_score", ascending=False).head(10)
)

In [ ]:
metrics_df[metrics_df["is_zero_demand"] == 1]["stockout_risk_score"].max()

In [ ]:
metrics_df[metrics_df["adjusted_reorder_gap"] > 0].shape

## **`scoring.py`**

Confirms that `load_and_build_scoring_df()` correctly applies the priority engine logic and that priority level distributions are consistent with expectations from the exploration phase.

In [ ]:
from engine.scoring import load_and_build_scoring_df

scoring_df = load_and_build_scoring_df()

print(scoring_df.shape)
display(scoring_df.head())

In [ ]:
display(scoring_df.sort_values("priority_rank", ascending=True).head(10))

In [ ]:
print(scoring_df["priority_level"].value_counts())

In [ ]:
print(scoring_df["needs_action"].value_counts())

In [ ]:
print(scoring_df[scoring_df["is_zero_demand"] == 1]["needs_action"].value_counts())
print(scoring_df[scoring_df["is_zero_demand"] == 1]["priority_level"].value_counts())

In [ ]:
display(
    scoring_df.sort_values(["priority_score", "priority_rank"], ascending=[False, True]).head(10)
)

In [ ]:
from engine.scoring import load_and_build_scoring_df, build_executive_view

scoring_df = load_and_build_scoring_df()
exec_df = build_executive_view(scoring_df)

display(exec_df.head(20))

In [ ]:
def generate_summary(df):
    total = len(df)
    critical = (df["priority_level"] == "CRITICAL").sum()
    high = (df["priority_level"] == "HIGH").sum()
    medium = (df["priority_level"] == "MEDIUM").sum()

    return {
        "total_items": total,
        "critical_items": critical,
        "high_items": high,
        "medium_items": medium,
        "action_required_pct": round(
            (df["needs_action"].sum() / total) * 100, 2
        ),
    }

generate_summary(exec_df)

## Validate Pipeline Output

Checks the shape, null counts, and priority distribution of the executive view dataframe to confirm that `build_executive_view` produces a complete, well-formed output with no unexpected nulls or label imbalances.

In [ ]:
print('Shape:', exec_df.shape)

print('\nNull counts:')
print(exec_df.isnull().sum())

print('\nPriority distribution:')
print(exec_df['priority_level'].value_counts())

print('\nNeeds action distribution:')
print(exec_df['needs_action'].value_counts())

distribusi = pd.crosstab(exec_df['priority_level'], exec_df['needs_action'])
print('\nPriority vs Needs Action:')
print(distribusi)

display(exec_df.head())


## Threshold & Distribution Checks

Examines the distribution of `days_of_stock_remaining` across the dataset to validate that the urgency signal is well-spread and that the chosen restock thresholds capture a meaningful proportion of SKUs.

In [ ]:
print('Days of stock remaining -- summary stats:')
print(exec_df['days_of_stock_remaining'].describe())

print('\nThreshold breakdown:')
for t in [1, 2, 3, 5, 7]:
    count = (exec_df['days_of_stock_remaining'] <= t).sum()
    pct = (exec_df['days_of_stock_remaining'] <= t).mean() * 100
    print(f'  <= {t} days: {count} SKU ({pct:.1f}%)')

print('\nLow-stock quantiles:')
print(exec_df['days_of_stock_remaining'].quantile([0.05, 0.10, 0.15, 0.20, 0.25]))

bins = [0, 1, 2, 3, 5, 7, 10, 15, 20, np.inf]
labels = ['0-1', '1-2', '2-3', '3-5', '5-7', '7-10', '10-15', '15-20', '20+']
coverage_dist = pd.cut(exec_df['days_of_stock_remaining'], bins=bins, labels=labels, right=True)
print('\nCoverage distribution:')
print(coverage_dist.value_counts().sort_index())


## Category Enrichment

Joins the `category` dimension from `products_df` onto the executive view so that downstream consumers (dashboard, Tableau) can filter and aggregate by product family without needing a separate lookup.

In [ ]:
product_mapping = products_df[['product_id', 'category']].copy()
product_mapping['product_id'] = product_mapping['product_id'].astype(str).str.strip()
exec_df['product_id'] = exec_df['product_id'].astype(str).str.strip()

dup_check = product_mapping.groupby('product_id')['category'].nunique().reset_index()
problem_ids = dup_check[dup_check['category'] > 1]
print('Products with duplicate category mappings:', len(problem_ids))

missing_products = set(exec_df['product_id']) - set(product_mapping['product_id'])
print('Products in exec_df missing from product_mapping:', len(missing_products))

enriched_df = exec_df.merge(
    product_mapping,
    on='product_id',
    how='left'
)

print('\nShape:', enriched_df.shape)
print('Unique categories:', enriched_df['category'].nunique())
print('Null category count:', enriched_df['category'].isna().sum())
print(enriched_df['category'].value_counts())


## Export Final Output

Writes the enriched executive view to `data/processed/` as the single authoritative output of this notebook -- the file consumed by the AI dashboard and any downstream reporting layers.

In [ ]:
from pathlib import Path

output_path = Path('../data/processed/executive_view_enriched.xlsx')
enriched_df.to_excel(output_path, index=False)
print('Saved to:', output_path)
